# 04 - Feature Selection (Simple MI Pipeline)

Step 1: Variance Threshold + Mutual Information on genes

Step 2: Domain-specific feature engineering (7 features)

Step 3: Combine selected genes + engineered + clinical features

All logic lives in `src/feature_selection.py`. This notebook only loads data, runs the pipeline, and saves artifacts.

In [1]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import joblib
import config
from src.io import logger
from src.feature_selection import run_feature_selection

## Step 1: Load Preprocessed Training Data

In [2]:
X_train = pd.read_csv(config.PROCESSED_DIR / "X_train_preprocessed.csv", index_col=0)

y_train_df = pd.read_csv(config.PROCESSED_DIR / "y_train.csv")
y_train = y_train_df.iloc[:, 0] if len(y_train_df.columns) == 1 else y_train_df["BCR"]

logger.info(f"Training data loaded: {X_train.shape}, Positives: {int(y_train.sum())}")

2026-09-01 09:51:01 | INFO     | prostate_bcr | Training data loaded: (343, 19018), Positives: 46


## Step 2: Run Feature Selection Pipeline

This performs:
1. Variance threshold on genes
2. Mutual Information to select top K genes
3. Creates 7 engineered features
4. Returns fitted selector + list of all selected features

In [3]:
clinical_keywords = [
    "gleason", "margin", "lymph", "tumor stage", "psa",
    "bone scan", "cause of death", "ct scan", "primary therapy",
    "age", "race", "ethnicity", "weight", "height",
    "mri", "icd-o", "histology", "patient primary", "diagnosis",
    "year cancer", "radical prostatectomy", "adjuvant", "radiation",
    "hormone", "chemotherapy", "surgery", "metastasis", "recurrence",
    "pathway_score", "_score", "risk", "total", "ratio", "balance",
]

clinical_cols = [
    c for c in X_train.columns
    if any(kw.lower() in c.lower() for kw in clinical_keywords)
]

logger.info(f"Detected {len(clinical_cols)} clinical/engineered columns to exclude from Layer 1.")
print(f"Clinical/engineered columns: {len(clinical_cols)}")
print(f"Gene columns:                {X_train.shape[1] - len(clinical_cols)}")

2026-09-01 09:51:01 | INFO     | prostate_bcr | Detected 132 clinical/engineered columns to exclude from Layer 1.


Clinical/engineered columns: 132
Gene columns:                18886


## Step 3: Run the 3-Layer Feature Selection Pipeline

Fitted exclusively on `X_train` / `y_train`.

In [4]:
# Cell for Feature Selection in Notebook 04
from src.feature_selection import run_3layer_feature_selection
import joblib
import os

print("🚀 Starting 3-Layer Feature Selection Pipeline...")
print(f"   Target Features: {config.PSO_FINAL_K} (Recommended: 40-45)")

# Ensure output directories exist
config.MODELS_DIR.mkdir(parents=True, exist_ok=True)
config.TABLES_DIR.mkdir(parents=True, exist_ok=True)

fitted_l1 = None
final_features = [] 

try:
    # FIX: Comprehensive clinical keywords including histology subtypes
    clinical_keywords = [
        'gleason', 'margin', 'lymph', 'tumor stage', 'psa', 
        'bone scan', 'cause of death', 'ct scan', 'primary therapy',
        'age', 'race', 'ethnicity', 'weight', 'height',
        'mri', 'icd-o', 'histology', 'patient primary', 'diagnosis',
        'year cancer', 'radical prostatectomy', 'adjuvant', 'radiation',
        'hormone', 'chemotherapy', 'surgery', 'metastasis', 'recurrence',
        'histologic', 'subtype', 'ductal', 'acinar'  # Added to fix mismatch error
    ]
    
    auto_clinical_cols = [
        c for c in X_train.columns 
        if any(kw.lower() in c.lower() for kw in clinical_keywords)
    ]
    
    engineered_keywords = ['pathway_score', '_score', 'risk', 'total', 'ratio', 'balance']
    auto_clinical_cols += [
        c for c in X_train.columns 
        if any(kw.lower() in c.lower() for kw in engineered_keywords) and c not in auto_clinical_cols
    ]
    
    print(f"   Detected {len(auto_clinical_cols)} clinical/engineered columns.")
    
    fitted_l1, final_features = run_3layer_feature_selection(
        X_train=X_train,
        y_train=y_train,
        clinical_cols=auto_clinical_cols,
        run_pso=True,
        random_state=config.RANDOM_STATE
    )

    print(f"\n Pipeline Complete!")
    print(f"   Final Total Features: {len(final_features)}")

except ValueError as e:
    if "feature names should match" in str(e).lower():
        print(f"\n CRITICAL ERROR: Column mismatch detected.")
        print(f"   Please verify 'clinical_keywords' list covers all non-gene columns.")
    else:
        raise

# # SAFE SAVE LOGIC (Outside try-except)
# if final_features:
#     joblib.dump(fitted_l1, config.MODELS_DIR / "fitted_layer1_selector.joblib")
#     pd.DataFrame({"feature": final_features}).to_csv(
#         config.TABLES_DIR / "selected_features_final.csv", index=False
#     )
#     print(f"   Saved artifacts successfully.")
# else:
#     print("\n⚠️ WARNING: No features generated. Artifacts NOT saved.")

🚀 Starting 3-Layer Feature Selection Pipeline...
   Target Features: 50 (Recommended: 40-45)
   Detected 146 clinical/engineered columns.


2026-09-01 09:51:32 | INFO     | prostate_bcr | Layer 1 - Selected 200 genes from 18872 raw genes
2026-09-01 09:51:33 | INFO     | prostate_bcr | Layer 1 - Transform: 200/200 genes available
2026-09-01 09:51:33 | INFO     | prostate_bcr | Layer 2 - Created 4 clean engineered features
2026-09-01 09:52:01 | INFO     | prostate_bcr |   PSO iter 5/10: Fitness=0.7582 (Raw AUC~=0.8082)
2026-09-01 09:52:24 | INFO     | prostate_bcr |   PSO iter 10/10: Fitness=0.7694 (Raw AUC~=0.8194)
2026-09-01 09:52:24 | INFO     | prostate_bcr | PSO selected 50 features
2026-09-01 09:52:24 | INFO     | prostate_bcr | 3-Layer Pipeline Complete: 50 final features



 Pipeline Complete!
   Final Total Features: 50


## Step 3: Save Artifacts

In [5]:
# Save fitted selector
joblib.dump(fitted_selector, config.MODELS_DIR / "fitted_selector.joblib")
print(f"Saved selector to: {config.MODELS_DIR / 'fitted_selector.joblib'}")

# Save feature list
pd.DataFrame({"feature": final_features}).to_csv(
    config.TABLES_DIR / "selected_features_final.csv", index=False
)
print(f"Saved feature list ({len(final_features)} items) to: {config.TABLES_DIR / 'selected_features_final.csv'}")

# Save engineered training data for reference
X_train_eng = X_train.copy()
for feat in eng_features:
    if feat not in X_train_eng.columns:
        X_train_eng[feat] = X_eng[feat]

print("Feature selection complete!")

Saved Layer 1 selector to: D:\Prostate_BCR\core\outputs\models\fitted_layer1_selector.joblib
Saved feature list (50 items) to: D:\Prostate_BCR\core\outputs\tables\selected_features_final.csv
